# Subdivision as re-factorisation — and why $\epsilon(C_6)\to\epsilon(C_7)$ is not creation

**A-Mark9 / NEXUS · arena programme · §11 · 2026-09-15**
Dean Kulik, QuHarmonics Research Group · ORCID 0009-0003-3128-8828

Self-contained: `itertools`, `numpy`, `collections` only.

---

## 1. The third mechanism

Two are measured: **extension** (add a base edge — the transformation set grows, $\Delta_R$ onto)
and **hiding** (enlarge the group — $R_{G'} \hookrightarrow R_G$, injective). Subdivision is
neither. A transformation is not added and not hidden; it is **re-factored**:

$$F\big([u_0 \to w_0]\ \text{at context } T\big) \;=\; [u_0 \to z] + [z \to w_0]\ \text{at the same } T$$

with every other $(e,T)$ carried across unchanged, and configurations included by
$V \subset V \cup \{z\}$. There is no inclusion of transformation sets, so the restriction
$\rho$ used for edge edits does not exist. What exists is a chain map.

$$\partial' F = F\,\partial \quad\text{(measured: } 0.0 \text{ exactly, all five cases)}$$

and dualising, $F^*: C^1(\mathcal A') \to C^1(\mathcal A)$ satisfies
$F^*(d_0 f') = d_0(f'|_{\mathcal A})$ because $F$ is the identity on configurations — so $F^*$
carries exact to exact and descends to residuals.

Common group: the stabiliser of the subdivided edge in $\operatorname{Aut}(X)$, extended to $X'$
by fixing $z$ (verified to land in $\operatorname{Aut}(X')$ in every case).

## 2. Library

In [1]:
import itertools, numpy as np
from collections import defaultdict
ATOL = 1e-9
def rank_abs(M):
    if M.size == 0 or min(M.shape) == 0: return 0
    return int((np.linalg.svd(M, compute_uv=False) > ATOL).sum())
def colspace(M):
    if M.size == 0 or min(M.shape) == 0: return np.zeros((M.shape[0], 0))
    U, s, _ = np.linalg.svd(M, full_matrices=False); return U[:, s > ATOL]

def autos(E, n):
    Es = set(map(tuple, map(sorted, E)))
    deg = defaultdict(int)
    for u, v in Es: deg[u] += 1; deg[v] += 1
    adj = defaultdict(set)
    for u, v in Es: adj[u].add(v); adj[v].add(u)
    out, p = [], [None]*n
    used = [False]*n
    def bt(i):
        if i == n: out.append(tuple(p)); return
        for c in range(n):
            if used[c] or deg[c] != deg[i]: continue
            ok = True
            for j in range(i):
                if (j in adj[i]) != (p[j] in adj[c]): ok = False; break
            if ok:
                p[i] = c; used[c] = True; bt(i+1); used[c] = False
    bt(0); return out

class Arena:
    def __init__(self, E, n, K):
        self.n, self.K = n, K
        self.E = sorted(map(tuple, map(sorted, E)))
        self.verts = sorted(itertools.combinations(range(n), K))
        self.vi = {frozenset(s): i for i, s in enumerate(self.verts)}
        self.items = [(frozenset(e), frozenset(T)) for e in self.E
                      for T in itertools.combinations([x for x in range(n) if x not in e], K-1)]
        self.idx = {it: i for i, it in enumerate(self.items)}
    def endpoints(self, it):
        e, T = it; u, w = sorted(e)
        return frozenset(T | {u}), frozenset(T | {w})     # oriented u -> w
    def d0(self):
        D = np.zeros((len(self.items), len(self.verts)))
        for it in self.items:
            A, B = self.endpoints(it)
            D[self.idx[it], self.vi[B]] += 1; D[self.idx[it], self.vi[A]] -= 1
        return D
    def boundary(self):                                    # partial_1 : C_1 -> C_0
        return self.d0().T
    def proj(self, G):
        P = np.zeros((len(self.items),)*2)
        for g in G:
            for it in self.items:
                e, T = it; u, w = sorted(e)
                j = (frozenset(g[i] for i in e), frozenset(g[i] for i in T))
                ju, jw = sorted(j[0])
                s = 1.0 if (g[u], g[w]) == (ju, jw) else -1.0
                P[self.idx[j], self.idx[it]] += s/len(G)
        return P
    def residual(self, G):
        P = self.proj(G); Inv = colspace(P); Ex = colspace(P @ colspace(self.d0()))
        return colspace(Inv - Ex @ (Ex.T @ Inv)) if Ex.shape[1] else Inv

def subdivide(E, n, e0):
    z = n
    Ep = [e for e in map(tuple, map(sorted, E)) if tuple(sorted(e0)) != e]
    u0, w0 = sorted(e0)
    return Ep + [(u0, z), (w0, z)], n+1, z

def chain_map(A, Ap, e0, z):
    """F : C_1(A) -> C_1(A'), and check d F = F d."""
    F = np.zeros((len(Ap.items), len(A.items)))
    u0, w0 = sorted(e0)
    for it in A.items:
        e, T = it
        if set(e) != {u0, w0}:
            F[Ap.idx[it], A.idx[it]] = 1.0
        else:
            F[Ap.idx[(frozenset((u0, z)), T)], A.idx[it]] += 1.0   # u0 -> z  (u0 < z)
            F[Ap.idx[(frozenset((w0, z)), T)], A.idx[it]] -= 1.0   # z -> w0  (canonical is w0 -> z)
    # vertex inclusion
    I = np.zeros((len(Ap.verts), len(A.verts)))
    for S, i in A.vi.items(): I[Ap.vi[S], i] = 1.0
    err = np.abs(Ap.boundary() @ F - I @ A.boundary()).max()
    return F, err

def run(name, E, n, K, e0):
    Ep, npr, z = subdivide(E, n, e0)
    A, Ap = Arena(E, n, K), Arena(Ep, npr, K)
    F, err = chain_map(A, Ap, e0, z)
    GX, GXp = autos(E, n), autos(Ep, npr)
    u0, w0 = sorted(e0)
    stab = [g for g in GX if {g[u0], g[w0]} == {u0, w0}]
    Gc  = stab
    Gcp = [tuple(list(g) + [z]) for g in stab]
    assert all(g in set(GXp) for g in Gcp), "extended stabiliser not in Aut(X')"
    RX, RXc = A.residual(GX), A.residual(Gc)
    RP, RPc = Ap.residual(GXp), Ap.residual(Gcp)
    # F* : C^1(A') -> C^1(A) is F^T ; descends to residuals
    M = RXc.T @ (F.T @ RPc) if (RXc.shape[1] and RPc.shape[1]) else np.zeros((RXc.shape[1], RPc.shape[1]))
    r = rank_abs(M)
    print(f"\n### {name}   K={K}   subdivide {tuple(sorted(e0))} -> new vertex {z}")
    print(f"    chain map: max |d F - F d| = {err:.1e}   ({'d F = F d' if err < 1e-12 else 'FAILS'})")
    print(f"    transformations: {len(A.items)} -> {len(Ap.items)}   "
          f"(survive verbatim {sum(1 for it in A.items if set(it[0])!={u0,w0})}, "
          f"re-factored into 2-paths {sum(1 for it in A.items if set(it[0])=={u0,w0})})")
    print(f"    |Aut(X)|={len(GX):<4} |Aut(X')|={len(GXp):<4} |common (edge stabiliser)|={len(Gc)}")
    print(f"    dim R:  X@Aut={RX.shape[1]:<3} X@common={RXc.shape[1]:<3} "
          f"X'@common={RPc.shape[1]:<3} X'@Aut={RP.shape[1]}")
    print(f"    F*_R : R_X'({RPc.shape[1]}) -> R_X({RXc.shape[1]})  rank={r}  "
          f"kernel={RPc.shape[1]-r} (created by the re-factoring)  cokernel={RXc.shape[1]-r} (lost)")
    # does the FULL-symmetry residual of X' survive pullback?
    if RP.shape[1]:
        img = F.T @ RP
        print(f"    full-Aut residual of X' pulled back: rank {rank_abs(img)} of {RP.shape[1]}; "
              f"component inside R_X@common = {rank_abs(RXc.T @ img) if RXc.shape[1] else 0}")

## 3. Runs


In [2]:
C = lambda m: [(i, (i+1) % m) for i in range(m)]
P = lambda m: [(i, i+1) for i in range(m-1)]
Q3 = [(v, v ^ (1 << i)) for v in range(8) for i in range(3) if v < v ^ (1 << i)]
run("C6 -> C7", C(6), 6, 3, (0, 1))
run("C7 -> C8", C(7), 7, 3, (0, 1))
run("C8 -> C9", C(8), 8, 3, (0, 1))
run("P6 -> P7 (middle edge)", P(6), 6, 3, (2, 3))
run("Q3 -> Q3 subdivided", Q3, 8, 3, (0, 1))



### C6 -> C7   K=3   subdivide (0, 1) -> new vertex 6
    chain map: max |d F - F d| = 0.0e+00   (d F = F d)
    transformations: 36 -> 70   (survive verbatim 30, re-factored into 2-paths 6)
    |Aut(X)|=12   |Aut(X')|=14   |common (edge stabiliser)|=2
    dim R:  X@Aut=0   X@common=7   X'@common=16  X'@Aut=1
    F*_R : R_X'(16) -> R_X(7)  rank=7  kernel=9 (created by the re-factoring)  cokernel=0 (lost)
    full-Aut residual of X' pulled back: rank 1 of 1; component inside R_X@common = 1

### C7 -> C8   K=3   subdivide (0, 1) -> new vertex 7
    chain map: max |d F - F d| = 0.0e+00   (d F = F d)
    transformations: 70 -> 120   (survive verbatim 60, re-factored into 2-paths 10)
    |Aut(X)|=14   |Aut(X')|=16   |common (edge stabiliser)|=2
    dim R:  X@Aut=1   X@common=16  X'@common=30  X'@Aut=2
    F*_R : R_X'(30) -> R_X(16)  rank=16  kernel=14 (created by the re-factoring)  cokernel=0 (lost)
    full-Aut residual of X' pulled back: rank 2 of 2; component inside R_X@common = 2

### 

## 4. Results

| subdivision | $\partial F - F\partial$ | transformations | $|G_X|,|G_{X'}|,|G_\cap|$ | $\dim R$: X@Aut / X@$\cap$ / X'@$\cap$ / X'@Aut | $\operatorname{rank}F^*_R$ | ker | **coker** |
|---|--:|---|---|---|--:|--:|--:|
| $C_6\to C_7$ | $0.0$ | 36 → 70 (30 survive, 6 re-factor) | 12, 14, 2 | 0 / 7 / 16 / 1 | 7 | 9 | **0** |
| $C_7\to C_8$ | $0.0$ | 70 → 120 (60, 10) | 14, 16, 2 | 1 / 16 / 30 / 2 | 16 | 14 | **0** |
| $C_8\to C_9$ | $0.0$ | 120 → 189 (105, 15) | 16, 18, 2 | 2 / 30 / 50 / 3 | 30 | 20 | **0** |
| $P_6\to P_7$ | $0.0$ | 30 → 60 (24, 6) | 2, 2, 2 | 5 / 5 / 12 / 12 | 5 | 7 | **0** |
| $Q_3\to Q_3'$ | $0.0$ | 180 → 273 (165, 15) | 48, 4, 4 | 1 / 26 / 42 / 42 | 26 | 16 | **0** |

$$\boxed{\ F^*_R \text{ is surjective in every case: cokernel } 0\ }$$

Subdivision joins extension: **no residual channel is lost when a transformation re-factors.**

### 4.1 The crucial one

$\epsilon(C_6,3)=0$ and $\epsilon(C_7,3)=1$. That looked like a loop being created by adding a
vertex. Pull the $C_7$ channel back:

| | $\dim R_{X',\operatorname{Aut}}$ | rank of $F^*$ on it | lands inside $R_{X,\cap}$ |
|---|--:|--:|--:|
| $C_6\to C_7$ | 1 | **1** | **1** |
| $C_7\to C_8$ | 2 | **2** | **2** |
| $C_8\to C_9$ | 3 | **3** | **3** |
| $P_6\to P_7$ | 12 | 10 | 5 |
| $Q_3\to Q_3'$ | 42 | 33 | 26 |

For the whole cycle family $F^*$ is **injective** on the full-symmetry residual: every channel of
the subdivided cycle pulls back to a nonzero cochain that already lives on the smaller cycle's
arena. It is $\mathbb Z_2$-invariant there and not $D_6$-invariant, which is exactly why
$\epsilon(C_6,3)$ reads $0$.

$$\boxed{\ \epsilon(C_6,3)=0 \to \epsilon(C_7,3)=1 \text{ is a change of visibility, not a creation}\ }$$

The same holds at $C_7 \to C_8 \to C_9$: the sequence $\epsilon(C_N,3) = 0,1,2,3$ for
$N = 6,7,8,9$ is not a sequence of loops appearing. The cochains are present throughout; the
dihedral group of the shorter cycle cannot see them.

The paths and $Q_3$ behave differently and the difference is informative: $2$ of $12$ and $9$ of
$42$ new channels are killed by $F^*$. Those are genuinely supported on the new structure — they
have no preimage in the old arena at all. So subdivision does both, and the two outcomes are
separable by one rank computation.

---

## 5. Where the three mechanisms now stand

$$\begin{array}{lll}
\textbf{extension} & \mathcal T_X \subset \mathcal T_{X'} & \Delta_R \text{ onto, coker } 0\\
\textbf{hiding} & G \subset G' & R_{G'} \hookrightarrow R_G,\ \ker 0\\
\textbf{re-factorisation} & T \mapsto T_2 T_1 & F^*_R \text{ onto, coker } 0
\end{array}$$

All three are one-sided exact, and in no case does a channel disappear. Restated with the
correction I owe — "nothing is destroyed" was too loose, because at full symmetry the drop
$R_{P_6} \cong \mathbb R^5 \to R_{C_6} = 0$ *is* real as a statement about invariant subspaces:

$$\boxed{\ \text{edits extend or re-factor; symmetry hides; projection erases visibility, not structure}\ }$$

### Not established

- Five subdivisions, $K=3$. Cokernel-$0$ looks like it should follow from $F$ being a chain map
  that is injective on configurations, but it is measured here, not proved.
- $F^*$ injective on the full-symmetry residual holds for the three cycles and fails for $P_6$ and
  $Q_3$. No characterisation of when it holds.
- Contraction, the adjoint operation, is untouched. It is the one that should be able to genuinely
  destroy, since it merges configurations rather than splitting transformations — and if it also
  turns out to be one-sided exact in the safe direction, the "nothing erases" statement gets much
  harder to dismiss.
- Nothing about mass, memory or matter. $R$ is a cochain sector.

## 6. Open

1. Contraction: $X' \to X$ by collapsing an edge. Expected to be the dual of subdivision, but the
   configuration map is many-to-one, so the induced object is a pushforward and may lose.
2. Prove the three one-sided exactness statements. All three look short; none is written.
3. Iterated edits: is $\operatorname{rank}$ of a composite $\Delta_R$ along a path through
   base-graph space the conserved quantity? That is the C1 question — what survives repeated
   change of substrate — and it is now well-posed for the first time.
